Data Collection




In [ ]:
!pip install feedparser pandas requests tqdm langdetect

import pandas as pd
import requests
import feedparser
import time
import urllib.parse
import random
import re
from getpass import getpass
from langdetect import detect, LangDetectException

# ============================================================
# CONFIG
# ============================================================

N_PER_DOMAIN = 200

# ============================================================
# YOUTUBE
# ============================================================

YOUTUBE_API_KEY = getpass("YouTube API key: ")

def collect_youtube(api_key, target=200):
    rows = []
    url = "https://www.googleapis.com/youtube/v3/videos"
    regions = ["US", "GB", "AU", "IE", "NZ"]

    for region in regions:
        if len(rows) >= target:
            break
        params = {
            "part": "snippet", "chart": "mostPopular",
            "maxResults": 50, "regionCode": region, "key": api_key
        }
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code != 200:
                print(f"YouTube {region}: {r.status_code}")
                continue
            for item in r.json().get("items", []):
                vid = item.get("id")
                title = item.get("snippet", {}).get("title")
                if title and vid:
                    rows.append({
                        "title": title.strip(), "domain": "youtube",
                        "source": "YouTube Data API", "source_id": vid,
                        "url": f"https://www.youtube.com/watch?v={vid}"
                    })
            print(f"YouTube {region}: {len(rows)}")
            time.sleep(0.2)
        except Exception as e:
            print(f"YouTube {region} failed: {e}")

    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset=["title"]).reset_index(drop=True)
    return df.head(target)

youtube_df = collect_youtube(YOUTUBE_API_KEY, N_PER_DOMAIN)
print(f"TOTAL YOUTUBE: {len(youtube_df)}")

# ============================================================
# REDDIT
# ============================================================

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
REDDIT_FEEDS = [
    "https://www.reddit.com/r/all/.rss",
    "https://www.reddit.com/r/all/top/.rss?t=all",
    "https://www.reddit.com/r/all/top/.rss?t=year",
    "https://www.reddit.com/r/all/top/.rss?t=month",
    "https://www.reddit.com/r/all/top/.rss?t=week",
    "https://www.reddit.com/r/all/top/.rss?t=day",
    "https://www.reddit.com/r/all/new/.rss",
    "https://www.reddit.com/r/all/rising/.rss",
    "https://www.reddit.com/r/all/controversial/.rss",
]

def collect_reddit(feeds, target=200, wait=65):
    rows = []
    seen = set()
    for i, url in enumerate(feeds):
        if len(rows) >= target:
            break
        print(f"Reddit feed {i+1}/{len(feeds)}")
        for attempt in range(3):
            try:
                r = requests.get(url, headers=HEADERS, timeout=15)
                if r.status_code == 200:
                    feed = feedparser.parse(r.content)
                    for entry in feed.entries:
                        t = entry.title.strip()
                        if t and t not in seen:
                            seen.add(t)
                            rows.append({
                                "title": t, "domain": "reddit",
                                "source": "r/all", "source_id": entry.id,
                                "url": entry.link
                            })
                    print(f"  +{len(rows)}")
                    break
                elif r.status_code == 429:
                    print(f"  429 — wait {wait}s")
                    time.sleep(wait)
                else:
                    break
            except Exception as e:
                print(f"  Failed: {e}")
                time.sleep(wait)
        if len(rows) < target and i < len(feeds) - 1:
            time.sleep(wait)
    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset=["title"]).reset_index(drop=True)
    return df.head(target)

reddit_df = collect_reddit(REDDIT_FEEDS, N_PER_DOMAIN)
print(f"TOTAL REDDIT: {len(reddit_df)}")

# ============================================================
# ARTICLES
# ============================================================

WORDS = ["the","and","for","are","but","not","you","all","can","had","her","was",
         "one","our","out","day","get","has","him","his","how","man","new","now",
         "old","see","two","way","who","boy","did","its","let","put","say","she",
         "too","use","that","with","have","this","will","your","from","they","know",
         "want","been","good","much","some","time","very","when","come","here","just",
         "like","long","make","many","more","only","over","such","take","than","them",
         "well","were","what","work","year","back","call","came","each","even","find",
         "give","hand","high","keep","last","left","life","live","look","made","most",
         "move","must","name","need","next","open","part","play","right","said","same",
         "seem","show","side","tell","turn","used","ways","week","went","word","world",
         "would","write"]

def collect_articles(words, target=200):
    rows = []
    random.shuffle(words)
    for w in words:
        if len(rows) >= target:
            break
        url = f"https://news.google.com/rss/search?q={urllib.parse.quote(w)}&hl=en-US&gl=US&ceid=US:en"
        try:
            feed = feedparser.parse(url)
            for entry in feed.entries:
                t = entry.get("title", "").strip()
                if t:
                    rows.append({
                        "title": t, "domain": "article",
                        "source": "Google News RSS", "source_id": "",
                        "url": entry.get("link", "")
                    })
            print(f"'{w}': {len(rows)}")
            time.sleep(0.3)
        except Exception as e:
            print(f"'{w}' failed: {e}")
    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset=["title"]).reset_index(drop=True)
    return df.head(target)

articles_df = collect_articles(WORDS, N_PER_DOMAIN)
print(f"TOTAL ARTICLES: {len(articles_df)}")

# ============================================================
# PODCASTS
# ============================================================

TERMS = ["life","story","talk","show","time","world","people","thing","way","day",
         "year","work","home","love","mind","body","health","money","power","change",
         "future","past","now","new","good","bad","big","small","high","low","first",
         "last","next","real","true","best"]

def collect_podcasts(terms, target=200):
    rows = []
    seen = set()
    random.shuffle(terms)
    for term in terms:
        if len(rows) >= target:
            break
        url = "https://itunes.apple.com/search"
        params = {"term": term, "media": "podcast", "limit": 50, "country": "US"}
        try:
            r = requests.get(url, params=params, timeout=30)
            for p in r.json().get("results", []):
                if len(rows) >= target:
                    break
                feed_url = p.get("feedUrl")
                name = p.get("collectionName", "")
                if not feed_url or name in seen:
                    continue
                seen.add(name)
                try:
                    feed = feedparser.parse(feed_url)
                    for entry in feed.entries[:5]:
                        t = entry.get("title", "").strip()
                        if t:
                            rows.append({
                                "title": t, "domain": "podcast",
                                "source": name, "source_id": entry.get("id", ""),
                                "url": entry.get("link", "")
                            })
                    time.sleep(0.1)
                except:
                    pass
            print(f"'{term}': {len(rows)}")
            time.sleep(0.3)
        except Exception as e:
            print(f"'{term}' failed: {e}")
    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset=["title"]).reset_index(drop=True)
    return df.head(target)

podcasts_df = collect_podcasts(TERMS, N_PER_DOMAIN)
print(f"TOTAL PODCASTS: {len(podcasts_df)}")

# ============================================================
# COMBINE + FILTER
# ============================================================

final_df = pd.concat([youtube_df, reddit_df, articles_df, podcasts_df], ignore_index=True)
final_df = final_df.drop_duplicates(subset=["title"]).reset_index(drop=True)

print(f"RAW: {len(final_df)}")
print(final_df["domain"].value_counts())

# Remove symbol-only
def has_words(t):
    if not isinstance(t, str): return False
    return len(re.findall(r"[a-zA-Z]", t)) >= 3
final_df = final_df[final_df["title"].apply(has_words)]

# English only
def is_en(t):
    try: return detect(str(t)) == "en"
    except: return False
final_df["en"] = final_df["title"].apply(is_en)
final_df = final_df[final_df["en"] == True].drop(columns=["en"])

# Min 3 words
final_df = final_df[final_df["title"].str.split().str.len() >= 3]

# Dedup again
final_df = final_df.drop_duplicates(subset=["title"]).reset_index(drop=True)

# New IDs
final_df.insert(0, "title_id", [f"T{i:04d}" for i in range(1, len(final_df)+1)])

print(f"\nFINAL: {len(final_df)}")
print(final_df["domain"].value_counts())

# Save
FINAL = final_df[["title_id","title","domain","source","source_id","url"]]
FINAL.to_csv("phase1_raw_titles.csv", index=False)
print("Saved phase1_raw_titles.csv")

In [1]:
import os
from google.colab import drive

# Mount Drive if not already mounted
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/ML Project/phase1_raw_titles.csv"

if os.path.exists(DRIVE_PATH):
    print(f"File already exists: {DRIVE_PATH}")
    print("Skipping save.")
else:
    FINAL.to_csv(DRIVE_PATH, index=False)
    print(f"Saved: {DRIVE_PATH}")
    print(f"Total titles: {len(FINAL)}")

Mounted at /content/drive
File already exists: /content/drive/MyDrive/ML Project/phase1_raw_titles.csv
Skipping save.


Annotation:

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

PROJECT_DIR = "/content/drive/MyDrive/ML Project/Final_ARI_Annotation"
DATA_DIR = os.path.join(PROJECT_DIR, "data")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, "annotation_output"), exist_ok=True)

DRIVE_PATH = "/content/drive/MyDrive/ML Project/phase1_raw_titles.csv"

df = pd.read_csv(DRIVE_PATH)

print("=" * 60)
print("LOADED DATA")
print("=" * 60)
print(f"Total: {len(df)}")
print(df["domain"].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
LOADED DATA
Total: 638
domain
article    199
podcast    172
reddit     160
youtube    107
Name: count, dtype: int64


In [3]:
domains = ["youtube", "reddit", "article", "podcast"]

pool_parts = []
for d in domains:
    subset = df[df["domain"] == d]
    n = min(80, len(subset))
    pool_parts.append(subset.sample(n=n, random_state=42))

annotation_pool = pd.concat(pool_parts).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Annotation pool: {len(annotation_pool)}")
print(annotation_pool["domain"].value_counts())

Annotation pool: 320
domain
article    80
youtube    80
reddit     80
podcast    80
Name: count, dtype: int64


In [4]:
common_counts = {"youtube": 8, "reddit": 8, "article": 7, "podcast": 7}

common_parts = []
for d in domains:
    subset = annotation_pool[annotation_pool["domain"] == d]
    n = min(common_counts[d], len(subset))
    common_parts.append(subset.sample(n=n, random_state=100))

common_set = pd.concat(common_parts).sample(frac=1, random_state=100).reset_index(drop=True)

print(f"Common set: {len(common_set)}")
print(common_set["domain"].value_counts())

Common set: 30
domain
reddit     8
youtube    8
podcast    7
article    7
Name: count, dtype: int64


In [5]:
remaining = annotation_pool[~annotation_pool["title_id"].isin(common_set["title_id"])]

unique_parts = []
for d in domains:
    subset = remaining[remaining["domain"] == d]
    n = min(25, len(subset))
    unique_parts.append(subset.sample(n=n, random_state=200))

unique_titles = pd.concat(unique_parts).reset_index(drop=True)

print(f"Unique titles: {len(unique_titles)}")
print(unique_titles["domain"].value_counts())

Unique titles: 100
domain
youtube    25
reddit     25
article    25
podcast    25
Name: count, dtype: int64


In [6]:
annotator_assignments = {}

for i in range(5):
    name = f"Annotator_{i+1}"
    parts = []

    for d in domains:
        du = unique_titles[unique_titles["domain"] == d].reset_index(drop=True)
        start = i * 5
        end = start + 5
        parts.append(du.iloc[start:end])

    au = pd.concat(parts)
    adf = pd.concat([common_set, au]).sample(frac=1, random_state=500+i).reset_index(drop=True)
    annotator_assignments[name] = adf

    print(f"{name}: {len(adf)} titles")

Annotator_1: 50 titles
Annotator_2: 50 titles
Annotator_3: 50 titles
Annotator_4: 50 titles
Annotator_5: 50 titles


In [7]:
for name, data in annotator_assignments.items():
    data.to_csv(os.path.join(DATA_DIR, f"{name}.csv"), index=False)
    data[["title_id"]].to_csv(os.path.join(DATA_DIR, f"{name}_instances.csv"), index=False)
    print(f"{name}: saved")

master = pd.concat(annotator_assignments.values()).drop_duplicates(subset="title_id").reset_index(drop=True)
master.to_csv(os.path.join(DATA_DIR, "annotation_master.csv"), index=False)

print(f"\nannotation_master.csv: {len(master)} titles")
print(f"Saved to: {DATA_DIR}")

Annotator_1: saved
Annotator_2: saved
Annotator_3: saved
Annotator_4: saved
Annotator_5: saved

annotation_master.csv: 130 titles
Saved to: /content/drive/MyDrive/ML Project/Final_ARI_Annotation/data


In [8]:
config_text = """
annotation_task_name: "What Makes a Title Interesting"
task_dir: "."
port: 8000

data_files:
  - "data/annotation_master.csv"

item_properties:
  id_key: "title_id"
  text_key: "title"

output_annotation_dir: "annotation_output/"
export_annotation_format:
  - "csv"

user_config:
  allow_all_users: false
  users:
    - "Annotator_1"
    - "Annotator_2"
    - "Annotator_3"
    - "Annotator_4"
    - "Annotator_5"

assignment_strategy: batch
batch_assignment:
  groups:
    - name: "annotator_1"
      annotators: ["Annotator_1"]
      instances_file: "data/Annotator_1_instances.csv"
    - name: "annotator_2"
      annotators: ["Annotator_2"]
      instances_file: "data/Annotator_2_instances.csv"
    - name: "annotator_3"
      annotators: ["Annotator_3"]
      instances_file: "data/Annotator_3_instances.csv"
    - name: "annotator_4"
      annotators: ["Annotator_4"]
      instances_file: "data/Annotator_4_instances.csv"
    - name: "annotator_5"
      annotators: ["Annotator_5"]
      instances_file: "data/Annotator_5_instances.csv"

max_annotations_per_user: 50

annotation_schemes:
  - annotation_type: "likert"
    name: "interest"
    description: "How likely would you be to consume this content based only on the title?"
    min_label: "Very unlikely"
    max_label: "Very likely"
    size: 6
    sequential_key_binding: true
    label_requirement:
      required: true
  - annotation_type: "multiselect"
    name: "clickbait_tactics"
    description: "Which tactics are present in this title? Select all that apply."
    labels:
      - "Missing details"
      - "Emotional"
      - "Exaggeration"
      - "Curiosity gap"
      - "Direct address"
      - "None"
    label_requirement:
      required: true
require_password: false
"""

with open(os.path.join(PROJECT_DIR, "config.yaml"), "w") as f:
    f.write(config_text)

print("Config saved")

Config saved


In [9]:
print("=" * 60)
print("FILES IN DATA FOLDER")
print("=" * 60)

for f in sorted(os.listdir(DATA_DIR)):
    path = os.path.join(DATA_DIR, f)
    size = os.path.getsize(path)
    print(f"{f} ({size} bytes)")

print("\nconfig.yaml exists:", os.path.exists(os.path.join(PROJECT_DIR, "config.yaml")))

FILES IN DATA FOLDER
Annotator_1.csv (12648 bytes)
Annotator_1_instances.csv (309 bytes)
Annotator_2.csv (12967 bytes)
Annotator_2_instances.csv (309 bytes)
Annotator_3.csv (12158 bytes)
Annotator_3_instances.csv (309 bytes)
Annotator_4.csv (12461 bytes)
Annotator_4_instances.csv (309 bytes)
Annotator_5.csv (12559 bytes)
Annotator_5_instances.csv (309 bytes)
annotation_master.csv (32477 bytes)

config.yaml exists: True


In [10]:
!pip install potato-annotation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 5.6 MB/s eta 0:00:00


In [11]:
%cd "/content/drive/MyDrive/ML Project/Final_ARI_Annotation"

!pkill -f potato
!sleep 2

!nohup potato start config.yaml --port 8000 > potato.log 2>&1 &
!sleep 5
!cat potato.log

/content/drive/MyDrive/ML Project/Final_ARI_Annotation
INFO: when you run the server directly from a .yaml file, please make sure your config file is put in the annotation project folder
2026-10-05 17:40:29 [INFO] potato.authentication: Initialized UserAuthenticator with method: in_memory, require_password: False
2026-10-05 17:40:30 [INFO] potato.item_state_management: ItemStateManager initialized with random_seed=1234
2026-10-05 17:40:30 [INFO] server_utils.front_end: Generating anntotation site at /usr/local/lib/python3.13/dist-packages/potato/templates
2026-10-05 17:40:30 [INFO] potato.server_utils.generated_templates: Generated templates directory: /usr/local/lib/python3.13/dist-packages/potato/templates/generated (the package directory is writable)
2026-10-05 17:40:30 [INFO] potato.server_utils.schemas.likert: Successfully generated likert layout for interest with 6 points
2026-10-05 17:40:30 [INFO] potato.server_utils.schemas.multiselect: Successfully generated multiselect layout

In [12]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!mv cloudflared /usr/local/bin/
!cloudflared --version

cloudflared version 2026.9.3 (built 2026-09-24-16:07 UTC)


In [13]:
import subprocess
import time
import re

process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Starting tunnel...")
time.sleep(15)

public_url = None
for _ in range(30):
    line = process.stdout.readline()
    if line:
        print(line.strip())
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            public_url = m.group(0)
            break

if public_url:
    print("\n" + "=" * 60)
    print("PUBLIC URL:", public_url)
    print("=" * 60)
else:
    print("\nNo URL found.")

Starting tunnel...
2026-10-05T17:40:34Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-05T17:40:34Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-05T17:40:38Z INF +--------------------------------------------------------------------------------------------+
2026-10-05T17:40:38Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-05T17:40:38Z INF |  https://graduate-mate-size-editor.